In [4]:
from itertools import product

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import seaborn as sns
from matplotlib.lines import Line2D
from pytensor.ifelse import ifelse
from scipy.optimize import minimize
from scipy.stats import genpareto
from tqdm.auto import tqdm

Let pandemic intensity $X$ be a random variable defined by the following cumulative distribution function:

$$
\mathrm{F_X}(x; p_0, \xi, \sigma) = \begin{cases}
     1 - p_0 & x < \mu \\
        1 - p_0\left(1+\xi\dfrac{x-\mu}{\sigma}\right)^{-1/\xi} & x \geq \mu
\end{cases}
$$

where $p_0$ is the annual probability that an epidemic emerges and $\xi$, $\mu$ and $\sigma$ are the shape, location, and scale parameters of a generalized Pareto distribution (GPD) representing pandemic intensity conditional on arrival. For simplicity we will henceforth denote $F_X(x;p_0, \xi, \sigma)$ as $F_X$.

The CEPI expert survey does not provide any information with which to calibrate our pandemic intensity prior distribution. We therefore adopt the common reference prior:

$$\pi(\xi, \theta, \lambda) \propto (1 + \xi)^{-1} \cdot \theta^{-1} \cdot \sqrt{\frac{1-p}{p}}$$

where $\theta = (1 + \xi)\cdot \sigma$.

Reference priors are attractive because "they let the data speak as much as possible." Formally, they maximize the expected Kullback-Leibler divergence between the prior and posterior (see [here](https://projecteuclid.org/journals/annals-of-statistics/volume-37/issue-2/The-formal-definition-of-reference-priors/10.1214/07-AOS587.full) for more).

The reparametrization in terms of $\theta$ is key. It yields a Fisher information matrix that guarantees the existence of a "common reference prior", i.e., one that is invariant to the sequence in which one conducts a marginalization procedure used in reference prior construction (see [Berger, Bernardo and Sun (2015)](https://projecteuclid.org/journals/bayesian-analysis/volume-10/issue-1/Overall-Objective-Priors/10.1214/14-BA915.full)). A derivation of the GPD common reference prior using this parametrization can be found in [Kang, Kim, and Lee (2013)](https://koreascience.or.kr/article/JAKO201306735656489.pdf). I posit that it is trivial to show that adding in arrival rate $p_0$ yields the same reference prior scaled by the Jeffrey's prior for $p_0$.


One downside of reference priors is that they are vulnerable to causing MCMC chains to diverge off into funnels when parameters yield extremes values in the prior. For our prior, this is more likely to occur when $\xi \to -1$ and $\theta \to 0$. We also run into issues as $\xi \to 0$, due to the $1 + \frac{1}{\xi}$ term in the likelihood function.

To tame this behavior, I've tried to implement [softplus functions](https://pytorch.org/docs/stable/generated/torch.nn.Softplus.html) that nudge the chain in the opposite direction at the very extremes. For $\xi$, I have more aggressively restricted its domain to $(0.1, \infty)$. We can justify this prior if we use the [Cirillo and Taleb log transformation](https://www.nature.com/articles/s41567-020-0921-x).

Another downside of the reference prior approach is that it's not analytically tractable if we wanted to add truncation from above in the form of $F_{\bar{u}}(x) = \frac{F(x)}{F(\bar{u})}$ as we have done before. Instead, we either have to use [Cirillo and Taleb's log transformation](https://www.nature.com/articles/s41567-020-0921-x) or sharp ex post truncation.

In [2]:
# Load actual duration data
final_allrisk_ds = pd.read_csv("../../data/clean/final_allrisk_ds.csv")
intensity_data = final_allrisk_ds.set_index('year_start')['intensity']

# Fill in zeros for years with no observations.
all_years = pd.Series(range(1900, 2023 + 1))
intensity_data = intensity_data.reindex(all_years, fill_value=0)

In [ ]:
THRESHOLD = 0.01

In [ ]:
class GPDBayesianUpdater:
    def __init__(self, prior_type='reference'):
        """
        Bayesian updater for GPD with reference prior.
        
        prior_type: only 'reference' is supported now.
        """
        if prior_type != 'reference':
            raise ValueError("Currently only 'reference' prior is implemented.")
        self.prior_type = prior_type


    def log_posterior(self, params, X, threshold):
        """
        Log posterior (up to a constant) for GPD with Jeffreys prior.
        """
        p, xi, sigma = params
        if sigma <= 0 or (xi <= -1):
            return -np.inf
        
        log_prior = -np.log(sigma) - 2 * np.log(1 + xi) + 0.5 * (np.log1p(-p) - np.log(p))
        
        is_tail = X > threshold
        tail_excess = X[is_tail] - threshold
        
        below_thresh_ll = np.sum(np.log1p(-p) * (~is_tail))
        above_thresh_ll = np.sum(np.log(p) + genpareto.logpdf(tail_excess, c=xi, scale=sigma))
        log_lik = below_thresh_ll + above_thresh_ll
        return log_lik + log_prior


    def update(self, X, threshold):
        """
        Perform posterior update given GPD exceedance data.
        """
        self.X = X

        ## Use MLE as initial guess for posterior mode
        is_tail = X > threshold
        tail_excess = X[is_tail] - threshold
        p_hat = np.mean(is_tail)
        
        c_hat, loc_hat, scale_hat = genpareto.fit(tail_excess, floc=0)
        res = minimize(lambda p: -self.log_posterior(p, X), 
                       x0=[c_hat, scale_hat],
                       method='L-BFGS-B',
                       bounds=[(-0.5, None), (1e-3, None)])  ## Bounds for xi and sigma
        
        self.p_map = p_hat
        self.xi_map, self.sigma_map = res.x
        self.optim_result = res


    def sample_posterior(self, n_samples=5000, proposal_scale_xi=1.5, proposal_scale_sigma=5, rng=None):
        """
        Sample from posterior using Metropolis-Hastings with boundary checking.
        Separate proposal scales for xi and sigma.
        """
        if not hasattr(self, 'xi_map'):
            raise ValueError("Call .update() before sampling.")

        rng = np.random.default_rng(rng)
        samples = []
        current = np.array([self.p_map, self.xi_map, self.sigma_map])
        current_logpost = self.log_posterior(current, self.X)

        accepted = 0
        for _ in range(n_samples):
            ## Propose new values for xi and sigma with different proposal scales
            proposal = current + np.array([
                rng.normal(scale=proposal_p_scale),
                rng.normal(scale=proposal_scale_xi),  ## Separate proposal scale for xi
                rng.normal(scale=proposal_scale_sigma)  ## Separate proposal scale for sigma
            ])

            ## Check if proposal is within bounds
            xi_proposal, sigma_proposal = proposal
            if sigma_proposal <= 0 or xi_proposal <= -1/2:
                continue  ## Skip invalid proposals

            logpost_proposal = self.log_posterior(proposal, self.X)

            ## Accept or reject the proposal based on the Metropolis-Hastings criterion
            if np.log(rng.uniform()) < logpost_proposal - current_logpost:
                current = proposal
                current_logpost = logpost_proposal
                accepted += 1  ## Count accepted proposal

            samples.append(current.copy())

        ## Display acceptance rate
        acceptance_rate = accepted / n_samples
        print(f"Acceptance rate: {acceptance_rate:.2%}")

        samples = np.array(samples)
        self.posterior_samples = samples
        return samples